# 03 · Exploratory schedule clustering

Run this after notebook 01. It compares KMeans and DBSCAN groupings using scheduled service features only. It does not train a supervised ridership predictor and does not classify routes as overcrowded.

The route-level passenger estimates are synthetic and are intentionally excluded from model inputs.

## Section 1 · Exploratory schedule clustering

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"
feature_files = sorted(PROCESSED_DIR.glob("route_features_*.csv"))
if not feature_files:
    raise FileNotFoundError("Run notebook 01 first; route_features_*.csv was not found.")
feature_path = feature_files[-1]
SERVICE_DATE = feature_path.stem.rsplit("_", 1)[-1]
features = pd.read_csv(feature_path, dtype={"route_id": str})
print(f"Feature file: {feature_path.name}")
print(f"Routes: {len(features):,}; service date: {SERVICE_DATE}")
print("Ridership field status:", features["ridership_data_status"].unique().tolist())

### 1. Build a schedule-only modeling matrix

In [ ]:
profile_columns = [f"trip_share_h{hour:02d}" for hour in range(24)]
SCHEDULE_FEATURES = ["scheduled_trips", "scheduled_hours", "service_span_hours",
                     "peak_trip_share", "distinct_stop_count", "median_stops_per_trip",
                     "median_trip_duration_min"] + profile_columns
X_df = features[SCHEDULE_FEATURES].apply(pd.to_numeric, errors="coerce")
X_df = X_df.replace([np.inf, -np.inf], np.nan).fillna(0)
scaler = StandardScaler()
X = scaler.fit_transform(X_df)
print(f"Model matrix: {X.shape[0]:,} routes × {X.shape[1]} schedule features")
print("Synthetic ridership columns used:", False)

### 2. Compare KMeans cluster counts

In [ ]:
if len(features) < 7:
    raise ValueError("At least seven route rows are required for this KMeans comparison.")

comparison = []
for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(X)
    score = silhouette_score(X, labels, sample_size=min(1500, len(X)), random_state=42)
    comparison.append({"k": k, "silhouette_sample": score, "inertia": model.inertia_,
                       "smallest_cluster_routes": int(pd.Series(labels).value_counts().min())})
comparison = pd.DataFrame(comparison).sort_values("k")
display(comparison)

best_k = int(comparison.loc[comparison["silhouette_sample"].idxmax(), "k"])
print(f"Exploratory selection by sampled silhouette: k={best_k}")

### 3. Fit the selected KMeans model and inspect clusters

A silhouette score only compares geometric separation under this feature scaling. It does not say that routes are efficient or that ridership was predicted accurately.

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(X)
centroid_distance = np.linalg.norm(X - kmeans.cluster_centers_[cluster_labels], axis=1)
clustered = features[["route_id", "route_short_name", "route_long_name", "service_date"]].copy()
clustered["schedule_cluster"] = cluster_labels
clustered["distance_to_cluster_center"] = centroid_distance

summary_base = features.copy()
summary_base["schedule_cluster"] = cluster_labels
cluster_summary = (summary_base.groupby("schedule_cluster")
                   .agg(route_count=("route_id", "nunique"),
                        median_scheduled_trips=("scheduled_trips", "median"),
                        median_operating_hours=("scheduled_hours", "median"),
                        median_distinct_stops=("distinct_stop_count", "median"),
                        median_trip_duration_min=("median_trip_duration_min", "median"),
                        median_peak_trip_share=("peak_trip_share", "median"))
                   .reset_index())
display(cluster_summary)

cluster_sizes = clustered["schedule_cluster"].value_counts().sort_index()
ax = cluster_sizes.plot(kind="bar", figsize=(8, 4), color="#247ba0", title="Routes per schedule-pattern cluster")
ax.set_xlabel("Schedule cluster")
ax.set_ylabel("Routes")
ax.grid(axis="y", alpha=.25)
plt.tight_layout()

### 4. Compare average departure-time profiles

In [ ]:
profiles = summary_base.groupby("schedule_cluster")[profile_columns].mean()
fig, ax = plt.subplots(figsize=(14, max(4, .8 * len(profiles))))
image = ax.imshow(profiles.to_numpy(), aspect="auto", cmap="Blues")
ax.set_yticks(range(len(profiles)))
ax.set_yticklabels([f"Cluster {i}" for i in profiles.index])
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45, ha="right")
ax.set_xlabel("Scheduled departure hour")
ax.set_title("Mean share of each route's trips by hour")
fig.colorbar(image, ax=ax, label="Mean within-route trip share")
plt.tight_layout()

### 5. Optional DBSCAN density check

In [ ]:
# Use the k-distance curve to choose an eps value; the setting below is an initial exploration only.
MIN_SAMPLES = 15
neighbors = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X)
distances, _ = neighbors.kneighbors(X)
k_distance = np.sort(distances[:, -1])
plt.figure(figsize=(10, 4))
plt.plot(k_distance, color="#f18f01")
plt.xlabel("Routes sorted by distance")
plt.ylabel(f"Distance to {MIN_SAMPLES}th nearest neighbor")
plt.title("DBSCAN k-distance diagnostic")
plt.grid(alpha=.25)
plt.tight_layout()

DBSCAN_EPS = 5.0  # Inspect the curve above and adjust before interpreting these groups.
dbscan_labels = DBSCAN(eps=DBSCAN_EPS, min_samples=MIN_SAMPLES).fit_predict(X)
dbscan_cluster_count = len(set(dbscan_labels) - {-1})
noise_share = float(np.mean(dbscan_labels == -1))
print(f"DBSCAN eps={DBSCAN_EPS:g}, min_samples={MIN_SAMPLES}")
print(f"Clusters excluding noise: {dbscan_cluster_count}; noise share: {noise_share:.1%}")
print("Treat this output as a sensitivity check; DBSCAN depends strongly on the selected eps.")

### 6. Export route cluster labels

In [ ]:
output_path = PROCESSED_DIR / f"route_schedule_clusters_{SERVICE_DATE}.csv"
clustered.to_csv(output_path, index=False)
print(f"Saved {len(clustered):,} route labels to {output_path}")

### Interpretation and limitations

- A cluster means routes have similar **scheduled trip volumes, operating spans, stop coverage, scheduled durations, and hourly supply profiles** under the chosen standardization.
- Cluster IDs are arbitrary; cluster 0 is not “best” or “worst.”
- KMeans' sampled silhouette is an internal geometry score, not an operational performance metric.
- DBSCAN's result is especially sensitive to `DBSCAN_EPS`; tune it with the diagnostic and document the chosen value.
- No clustering result from this notebook establishes actual ridership or overcrowding. For those outcomes, first acquire route-level counts or a documented survey.